In [35]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [36]:
from torchgeo_bench.coordbench import probe, probe_optimized
from torchgeo_bench.coordbench.datasets import load_air_temp, load_chelsa
from torchgeo_bench.coordbench.models import SinCosLocationEncoder, ClimplicitLocationEncoder, TemporalSatCLIPEncoder
from torchgeo_bench.coordbench.probe import RIDGE_ALPHAS, _fold_indices, _valid_mask
from torchgeo_bench.coordbench.splits import spatial_fold_ids


# Try on Air Temperature

Try first just with naive SinCos location encoder

In [37]:
bench = load_air_temp()[0]
labels = bench.tasks["air_temperature"]
features = SinCosLocationEncoder().encode(bench.lon, bench.lat, None)
fold_assign = spatial_fold_ids(bench.lat, bench.lon, 5, 10.0, 0)

In [38]:
v = _valid_mask(features, labels, "regression")
X, y = features[v], labels[v]
folds = _fold_indices(len(X), 5, 0, fold_assign[v])
splits = [(np.setdiff1d(np.arange(len(X)), te), te) for te in folds]

In [39]:
for name, mod in [("probe", probe), ("probe_optimized", probe_optimized)]:
    score, _ = mod.linear_probe_score(
        features, labels, "regression", fold_assign=fold_assign, device="cpu"
    )
    print(f"{name:16s} R2 = {score:.6f}")

probe            R2 = 0.842056
probe_optimized  R2 = 0.842075


/home/libe2152/projects/torchgeo-bench/src/torchgeo_bench/coordbench/probe.py:208: UserWarning: ridge alpha selected at grid edge (0.0001); widen RIDGE_ALPHAS
  _, fold_scores = _cv_alpha_scores(data, fold_ids, alphas, standardize=standardize)


In [40]:
gs = GridSearchCV(
    make_pipeline(StandardScaler(), Ridge()),
    {"ridge__alpha": list(RIDGE_ALPHAS)},
    cv=splits,
    scoring="r2",
).fit(X, y)
print(f"{'sklearn':16s} R2 = {gs.best_score_:.6f}  (alpha={gs.best_params_['ridge__alpha']:g})")


sklearn          R2 = 0.842075  (alpha=10)


# Try on CHELSA dataset

### With Climplicit embeddings for one-day dataset

In [ ]:
bench = load_chelsa()[0]
day = bench.posix_timestamp == np.unique(bench.posix_timestamp)[180]
lat, lon, ts = bench.lat[day], bench.lon[day], bench.posix_timestamp[day]
labels = bench.tasks["tas"][day]
features = ClimplicitLocationEncoder().encode(lon, lat, ts)
fold_assign = spatial_fold_ids(lat, lon, 5, 10.0, 0)

In [42]:
v = _valid_mask(features, labels, "regression")
X, y = features[v], labels[v]
folds = _fold_indices(len(X), 5, 0, fold_assign[v])
splits = [(np.setdiff1d(np.arange(len(X)), te), te) for te in folds]

In [43]:
for name, mod in [("probe", probe), ("probe_optimized", probe_optimized)]:
    score, _ = mod.linear_probe_score(features, labels, "regression",
                                      fold_assign=fold_assign, device="cpu")
    print(f"{name:16s} R2 = {score:.6f}")

probe            R2 = 0.275053
probe_optimized  R2 = 0.398321


In [44]:
gs = GridSearchCV(make_pipeline(StandardScaler(), Ridge()),
                  {"ridge__alpha": list(RIDGE_ALPHAS)}, cv=splits, scoring="r2").fit(X, y)
print(f"{'sklearn':16s} R2 = {gs.best_score_:.6f}  (alpha={gs.best_params_['ridge__alpha']:g})")

sklearn          R2 = 0.398328  (alpha=316.228)


### Compare with Temporal SatCLIP embeddings and multi-day datasets

In [45]:
from torchgeo_bench.coordbench.temporal_aggregation import temporal_aggregation

In [46]:
def compare(features, labels, lat, lon, folds=5, seed=0):
    fold_assign = spatial_fold_ids(lat, lon, folds, 10.0, seed)
    v = _valid_mask(features, labels, "regression")
    X, y = features[v], labels[v]
    fids = _fold_indices(len(X), folds, seed, fold_assign[v])
    splits = [(np.setdiff1d(np.arange(len(X)), te), te) for te in fids]

    out = {}
    for name, mod in [("probe", probe), ("probe_optimized", probe_optimized)]:
        out[name] = mod.linear_probe_score(
            features, labels, "regression", folds=folds, seed=seed,
            fold_assign=fold_assign, device="cpu")[0]
    gs = GridSearchCV(make_pipeline(StandardScaler(), Ridge()),
                      {"ridge__alpha": list(RIDGE_ALPHAS)}, cv=splits, scoring="r2").fit(X, y)
    out["sklearn"] = gs.best_score_
    out["sklearn_alpha"] = gs.best_params_["ridge__alpha"]
    return out

In [47]:
bench = load_chelsa()[0]
TASK = "tas"

encoders = {
    "climplicit": ClimplicitLocationEncoder(),
    "t-satclip": TemporalSatCLIPEncoder(
        ckpt_path="/home/libe2152/projects/temporal-satclip/ckpts/tsatclip-doy-1M.ckpt",
        repo_path="/home/libe2152/projects/temporal-satclip",
        model_name="tsatclip/doy",
    ),
}

using pretrained moco vit16
using loss function: satclip_loss
using temporal model: True


In [ ]:
for enc_name, enc in encoders.items():
    # single day: one timestamp, ~10k independent locations
    day = bench.posix_timestamp == np.unique(bench.posix_timestamp)[180]
    lat, lon, ts = bench.lat[day], bench.lon[day], bench.posix_timestamp[day]
    feats = enc.encode(lon, lat, ts)
    print(enc_name, "single-day", compare(feats, bench.tasks[TASK][day], lat, lon))

    # multiday: labels/embeddings averaged over windows (rows = window x location)
    wb, emb = temporal_aggregation(bench, "4_week", encoder=enc)   # for now 4_week
    print(enc_name, "4_week", compare(emb, wb.tasks[TASK], wb.lat, wb.lon))

Output()

climplicit single-day {'probe': 0.2750533101503788, 'probe_optimized': 0.39832076886411677, 'sklearn': np.float64(0.39832847118377684), 'sklearn_alpha': 316.22776601683796}


climplicit 4_week {'probe': 0.5303918438300907, 'probe_optimized': 0.5303950867265487, 'sklearn': np.float64(0.5303943514823913), 'sklearn_alpha': 1.0}


/home/libe2152/projects/torchgeo-bench/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.3534644505728295e-10.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/home/libe2152/projects/torchgeo-bench/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 6.16794781915786e-10.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/home/libe2152/projects/torchgeo-bench/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.3700642009784048e-10.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/home/libe2152/projects/torchgeo-bench/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 7.209605829

Output()

/home/libe2152/projects/torchgeo-bench/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 8.53561132885261e-08.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


t-satclip single-day {'probe': 0.9082594912268138, 'probe_optimized': 0.9082522407497349, 'sklearn': np.float64(0.9082216620445251), 'sklearn_alpha': 0.03162277660168379}


/home/libe2152/projects/torchgeo-bench/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.2081040601816824e-10.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/home/libe2152/projects/torchgeo-bench/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.47150308499522e-10.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/home/libe2152/projects/torchgeo-bench/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 2.991170644506269e-10.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/home/libe2152/projects/torchgeo-bench/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 4.0825237634

t-satclip 4_week {'probe': 0.8998245260712988, 'probe_optimized': 0.8998256224016566, 'sklearn': np.float64(0.899801230430603), 'sklearn_alpha': 0.31622776601683794}


/home/libe2152/projects/torchgeo-bench/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 7.968737669727943e-09.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
